# Day 15: CNN Basics
Today we will cover:
- Convolutions
- Pooling
- Activation Functions

## Why CNNs?
Convolutional Neural Networks (CNNs) are designed to automatically and adaptively learn spatial hierarchies of features from images. They are the foundation of modern computer vision.

## Convolutions
- Apply filters (kernels) to extract features.
- Detect edges, textures, and patterns.

## Pooling
- Downsamples feature maps.
- Reduces computation and controls overfitting.

## Activation Functions
- Introduce non-linearity.
- Common: ReLU, Sigmoid, Tanh.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transform: normalize MNIST digits
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST dataset
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

100.0%
100.0%
100.0%
100.0%


## Exercise: Implement CNN in PyTorch for MNIST

In [2]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

model = CNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training Loop

In [3]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx*len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
for epoch in range(1, 3):
    train(model, device, train_loader, optimizer, epoch)

Train Epoch: 1 [0/60000] Loss: 2.315268
Train Epoch: 1 [6400/60000] Loss: 0.219588
Train Epoch: 1 [12800/60000] Loss: 0.121863
Train Epoch: 1 [19200/60000] Loss: 0.096339
Train Epoch: 1 [25600/60000] Loss: 0.116924
Train Epoch: 1 [32000/60000] Loss: 0.205850
Train Epoch: 1 [38400/60000] Loss: 0.045340
Train Epoch: 1 [44800/60000] Loss: 0.157936
Train Epoch: 1 [51200/60000] Loss: 0.212695
Train Epoch: 1 [57600/60000] Loss: 0.148813
Train Epoch: 2 [0/60000] Loss: 0.126069
Train Epoch: 2 [6400/60000] Loss: 0.112089
Train Epoch: 2 [12800/60000] Loss: 0.231523
Train Epoch: 2 [19200/60000] Loss: 0.029764
Train Epoch: 2 [25600/60000] Loss: 0.168156
Train Epoch: 2 [32000/60000] Loss: 0.065408
Train Epoch: 2 [38400/60000] Loss: 0.059376
Train Epoch: 2 [44800/60000] Loss: 0.110130
Train Epoch: 2 [51200/60000] Loss: 0.021588
Train Epoch: 2 [57600/60000] Loss: 0.006699


## Testing Loop

In [4]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'Test set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)')

test(model, device, test_loader)

Test set: Average loss: 0.0374, Accuracy: 9879/10000 (98.79%)


## Mini Project: Digit Classifier
We now have a trained CNN that can classify handwritten digits from the MNIST dataset. Try feeding custom digit images or drawing digits to test the classifier.